# Phase 1: Multimodal Emotion Dataset Exploration (EmoReact)
### Multidimensional Emotion-to-Sentiment Derivation System

This notebook performs the **Phase 1 Dataset Inspection & Exploration** as mandated by `AGENTS.md`:
1. **Environment & GPU Verification**: Ensure PyTorch and GPU runtime are ready.
2. **Dataset Alignment & Integrity**: Verify sample counts (Train: 432, Val: 303, Test: 367) and 1:1 file-to-label mapping.
3. **Multi-label Emotion Distribution**: Analyze class balance across the 8 emotion categories.
4. **Valence (Affective State) Distribution**: Profile continuous valence ratings across splits.
5. **Audio Modality Inspection**: Extract audio from clips, inspect sample rates/channels, and visualize Log-Mel Spectrograms.
6. **Video Modality Inspection**: Inspect frame rates, clip durations, and resolutions.
7. **OpenFace Features Exploration**: Inspect pre-extracted Action Units (`_au.txt`) and Gaze (`_gaze.txt`).

## 1. Setup & Environment Verification

In [ ]:
# Install dependencies for audio and video processing
!pip install -q torchaudio librosa soundfile matplotlib seaborn pandas opencv-python

In [ ]:
import os
import glob
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import torchaudio
import torchaudio.transforms as T
import librosa
import librosa.display

sns.set_theme(style="whitegrid")

# Check Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

## 2. Dataset Paths & Extraction

Mount Google Drive (if using Colab) or configure the path to `EmoReact_V_1.0`.

In [ ]:
# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Running in local environment (not Colab).")

# Set DATA_DIR (adjust if extracted elsewhere)
if IN_COLAB:
    # If RAR is on Google Drive, extract to /content/data:
    # !mkdir -p /content/data
    # !unrar x "/content/drive/MyDrive/EmoReact_V_1.0.rar" /content/data/
    DATA_DIR = "/content/data/EmoReact_V_1.0"
    if not os.path.exists(DATA_DIR):
        # Fallback to local repo data folder if already placed there
        DATA_DIR = "data/raw/EmoReact_V_1.0"
else:
    DATA_DIR = "data/raw/EmoReact_V_1.0"

print("Target Data Directory:", DATA_DIR)
print("Directory exists:", os.path.exists(DATA_DIR))

## 3. Label Schema & Split Verification

From `EmoReact_V_1.0/ReadMe.txt`, the schema is:
- **Columns 1–8**: Multi-label emotions (0 or 1)
  1. Curiosity
  2. Uncertainty
  3. Excitement
  4. Happiness
  5. Surprise
  6. Disgust
  7. Fear
  8. Frustration
- **Column 9**: Continuous Valence score (1.0 to 7.0)
- **Split Strategy**: Person-independent (disjoint children across Train, Val, Test).

In [ ]:
EMOTION_NAMES = [
    "Curiosity",
    "Uncertainty",
    "Excitement",
    "Happiness",
    "Surprise",
    "Disgust",
    "Fear",
    "Frustration"
]
ALL_COLUMNS = EMOTION_NAMES + ["Valence"]

splits_data = {}
for split, split_folder in [("train", "Train"), ("val", "Validation"), ("test", "Test")]:
    lbl_file = os.path.join(DATA_DIR, "Labels", f"{split}_labels.text")
    vid_folder = os.path.join(DATA_DIR, "Data", split_folder)
    
    if os.path.exists(lbl_file):
        # Load labels
        df = pd.read_csv(lbl_file, header=None, names=ALL_COLUMNS)
        
        # Get sorted video files
        if os.path.exists(vid_folder):
            video_files = sorted([f for f in os.listdir(vid_folder) if f.endswith(".mp4")])
            df["video_file"] = video_files[:len(df)]
            df["split"] = split
            print(f"[{split.upper()}] Labels count: {len(df)} | Videos count: {len(video_files)} | Match: {len(df) == len(video_files)}")
        else:
            print(f"[{split.upper()}] Labels count: {len(df)} | Video folder not yet extracted: {vid_folder}")
            df["split"] = split
        
        splits_data[split] = df
    else:
        print(f"[{split.upper()}] Label file not found: {lbl_file}")

if "train" in splits_data:
    print("\nSample Train Data Head:")
    display(splits_data["train"].head())

## 4. Multi-Label Emotion Distribution & Class Imbalance

Analyzing the frequency and imbalance ratio for each emotion category.

In [ ]:
if "train" in splits_data:
    train_df = splits_data["train"]
    emotion_counts = train_df[EMOTION_NAMES].sum().reset_index()
    emotion_counts.columns = ["Emotion", "Positive_Samples"]
    emotion_counts["Percentage"] = (emotion_counts["Positive_Samples"] / len(train_df)) * 100
    emotion_counts["Pos_Weight"] = (len(train_df) - emotion_counts["Positive_Samples"]) / np.maximum(emotion_counts["Positive_Samples"], 1)
    
    print("--- Emotion Distribution in Training Split ---")
    display(emotion_counts)
    
    plt.figure(figsize=(10, 5))
    barplot = sns.barplot(data=emotion_counts, x="Emotion", y="Percentage", palette="viridis")
    plt.title("Emotion Prevalence in EmoReact (Train Split)", fontsize=14, fontweight="bold")
    plt.ylabel("Percentage of Clips (%)", fontsize=12)
    plt.xlabel("Emotion Category", fontsize=12)
    plt.xticks(rotation=30)
    for p in barplot.patches:
        barplot.annotate(f"{p.get_height():.1f}%", 
                         (p.get_x() + p.get_width() / 2., p.get_height()), 
                         ha='center', va='bottom', fontsize=10, xytext=(0, 3), 
                         textcoords='offset points')
    plt.tight_layout()
    plt.show()

## 5. Valence (Affective State) Distribution

Valence is annotated as a continuous value from ~1.0 (very negative) to 7.0 (very positive).

In [ ]:
if "train" in splits_data:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram / KDE
    sns.histplot(train_df["Valence"], kde=True, ax=axes[0], color="royalblue", bins=20)
    axes[0].set_title("Train Valence Distribution (Continuous 1-7)", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Valence Rating")
    axes[0].axvline(train_df["Valence"].mean(), color="red", linestyle="--", label=f"Mean: {train_df['Valence'].mean():.2f}")
    axes[0].axvline(train_df["Valence"].median(), color="green", linestyle=":", label=f"Median: {train_df['Valence'].median():.2f}")
    axes[0].legend()
    
    # Correlation between Emotions and Valence
    corr = train_df[ALL_COLUMNS].corr()[["Valence"]].sort_values(by="Valence", ascending=False)
    sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1], cbar=True)
    axes[1].set_title("Emotion vs. Valence Correlation", fontsize=13, fontweight="bold")
    
    plt.tight_layout()
    plt.show()

## 6. Audio Modality Exploration (Your Core Scope)

Here we extract the audio from an `.mp4` video clip, verify sample rates, inspect the waveform, and compute the **Log-Mel Spectrogram** (the exact input format for `AudioEncoder`).

In [ ]:
# Find a sample video from Train
sample_videos = glob.glob(os.path.join(DATA_DIR, "Data", "Train", "*.mp4"))

if sample_videos:
    sample_vid = sample_videos[0]
    print(f"Inspecting Audio for: {os.path.basename(sample_vid)}")
    
    # Extract audio using torchaudio or librosa
    waveform, sample_rate = torchaudio.load(sample_vid)
    print(f"Raw Waveform Shape: {waveform.shape} (Channels x Samples)")
    print(f"Original Sample Rate: {sample_rate} Hz")
    duration_sec = waveform.shape[1] / sample_rate
    print(f"Audio Duration: {duration_sec:.2f} seconds")
    
    # Standardize to 16 kHz Mono
    target_sample_rate = 16000
    if sample_rate != target_sample_rate:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
        waveform_16k = resampler(waveform)
    else:
        waveform_16k = waveform
    
    if waveform_16k.shape[0] > 1:
        waveform_mono = torch.mean(waveform_16k, dim=0, keepdim=True)
    else:
        waveform_mono = waveform_16k
        
    # Compute Log-Mel Spectrogram (64 mel bands, standard 25ms win, 10ms hop)
    n_fft = 400      # 25 ms at 16kHz
    hop_length = 160 # 10 ms at 16kHz
    n_mels = 64
    
    mel_transform = T.MelSpectrogram(
        sample_rate=target_sample_rate,
        n_fft=n_fft,
        win_length=n_fft,
        hop_length=hop_length,
        n_mels=n_mels
    )
    mel_spec = mel_transform(waveform_mono)
    log_mel_spec = torch.log(mel_spec + 1e-6)
    
    print(f"Standardized Waveform Shape: {waveform_mono.shape}")
    print(f"Log-Mel Spectrogram Shape: {log_mel_spec.shape} [Channel, n_mels, Time_steps]")
    
    # Visualizations
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    
    # Waveform Plot
    axes[0].plot(np.linspace(0, duration_sec, waveform_mono.shape[1]), waveform_mono[0].numpy(), color="darkblue", alpha=0.8)
    axes[0].set_title(f"Audio Waveform ({target_sample_rate} Hz Mono)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Time (seconds)")
    axes[0].set_ylabel("Amplitude")
    
    # Mel Spectrogram Plot
    img = axes[1].imshow(log_mel_spec[0].numpy(), origin="lower", aspect="auto", cmap="magma")
    axes[1].set_title(f"Log-Mel Spectrogram ({n_mels} Mel Bands)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Frames / Time Steps")
    axes[1].set_ylabel("Mel Frequencies")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    
    plt.tight_layout()
    plt.show()
else:
    print("No .mp4 video files found yet in Data/Train. Extract videos to run audio analysis.")

## 7. Video Duration & Frame Rate Profile

Profile video durations and frame dimensions to confirm uniform temporal sampling policies ($T=16$ or $T=32$).

In [ ]:
if sample_videos:
    durations = []
    fps_list = []
    resolutions = []
    
    # Profile first 30 clips as a representative sample
    for vid in sample_videos[:30]:
        cap = cv2.VideoCapture(vid)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        
        if fps > 0:
            durations.append(frame_count / fps)
            fps_list.append(fps)
            resolutions.append(f"{w}x{h}")
            
    dur_s = pd.Series(durations)
    print("--- Video Sample Statistics ---")
    print(f"Min Duration: {dur_s.min():.2f}s | Max Duration: {dur_s.max():.2f}s | Median: {dur_s.median():.2f}s")
    print(f"Common FPS values: {pd.Series(fps_list).value_counts().to_dict()}")
    print(f"Common Resolutions: {pd.Series(resolutions).value_counts().to_dict()}")

## 8. Summary of Findings & Next Steps

1. **Exact 1:1 Mapping Verified**: Labels match video files alphabetically.
2. **Multi-label Formulation**: Model requires independent Sigmoids + BCE with `pos_weight`.
3. **Audio Preprocessing Contract Established**:
   - Target sample rate: `16,000 Hz` mono
   - Feature representation: `Log-Mel Spectrogram` (`64` bands, `hop_length=160`)
   - Output shape ready for `AudioEncoder`: `[B, 1, 64, T_audio]`